In [0]:

from pyspark.sql.functions import (
    col,
    trim,
    regexp_replace,
    concat_ws,
    length
)

BRONZE_TABLE = "career_os.bronze.jobs_raw"
SILVER_TABLE = "career_os.silver.jobs_clean"

bronze_df = spark.table(BRONZE_TABLE)
silver_df = bronze_df.withColumn(
    "clean_description",
    trim(
        regexp_replace(col("description"),"\\s+"," ")
    )
)

silver_df = silver_df.withColumn(
    "search_text",
    concat_ws(
        " ",
        col("title"),
        col("company"),
        col("location"),
        col("clean_description")
    )
)

silver_df = silver_df.withColumn("description_length",length(col("clean_description")))

(
    silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        SILVER_TABLE
    )
)